In [0]:
from pyspark.sql.functions import(col, concat, lit, to_timestamp, to_date,try_to_timestamp, concat_ws,trim, upper, lower )

In [0]:
%run "../includes/common_functions"


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS f1.silver;

In [0]:
qualifying_df = spark.read.table("f1.bronze.qualifying")

In [0]:
qualifying_renamed_df = qualifying_df \
    .withColumnRenamed("qualifyId", "qualifying_id") \
    .withColumnRenamed("raceId", "race_id") \
    .withColumnRenamed("driverId", "driver_id") \
    .withColumnRenamed("constructorId", "constructor_id")

In [0]:
qualifying_clean_df = qualifying_renamed_df \
    .filter(col("qualifying_id").isNotNull()) \
    .filter(col("race_id").isNotNull()) \
    .filter(col("driver_id").isNotNull()) \
    .dropDuplicates(["qualifying_id"])

In [0]:
qualifying_final_df = add_ingestion_date(qualifying_clean_df) \
    .withColumn("environment", lit("production")) \
    .withColumn("file_date", to_date(lit("2026-05-07")))

In [0]:
qualifying_final_df.createOrReplaceTempView("qualifying_updates")

In [0]:
if spark.catalog.tableExists("f1.silver.qualifying"):

    spark.sql("""

    MERGE INTO f1.silver.qualifying tgt
    USING qualifying_updates src
    ON tgt.qualifying_id = src.qualifying_id

    WHEN MATCHED THEN
    UPDATE SET *

    WHEN NOT MATCHED THEN
    INSERT *

    """)

else:

    qualifying_final_df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("f1.silver.qualifying")

In [0]:
%sql

SELECT *
FROM f1.silver.qualifying
LIMIT 10;